In [1]:
import pickle

In [2]:
with open('/root/autodl-tmp/chuandian_eq/data/taxi/raw/dev.pkl', 'rb') as f:
    data = pickle.load(f)

In [3]:
print(data['dev'][0])
print(len(data['dev']))
print(data['dim_process'])

[{'idx_event': 1, 'type_event': 5, 'time_since_start': 0.0, 'time_since_last_event': 0.0}, {'idx_event': 2, 'type_event': 1, 'time_since_start': 0.4363888888888889, 'time_since_last_event': 0.4363888888888889}, {'idx_event': 3, 'type_event': 5, 'time_since_start': 1.1122222222222222, 'time_since_last_event': 0.6758333333333333}, {'idx_event': 4, 'type_event': 0, 'time_since_start': 1.4241666666666666, 'time_since_last_event': 0.31194444444444436}, {'idx_event': 5, 'type_event': 5, 'time_since_start': 1.8616666666666666, 'time_since_last_event': 0.4375}, {'idx_event': 6, 'type_event': 3, 'time_since_start': 2.2730555555555556, 'time_since_last_event': 0.411388888888889}, {'idx_event': 7, 'type_event': 8, 'time_since_start': 3.278888888888889, 'time_since_last_event': 1.0058333333333334}, {'idx_event': 8, 'type_event': 3, 'time_since_start': 3.4175, 'time_since_last_event': 0.13861111111111102}, {'idx_event': 9, 'type_event': 8, 'time_since_start': 3.460277777777778, 'time_since_last_eve

In [4]:
from src.data.sequence import Sequence,EventSequence
import torch
def list_of_dicts_to_sequence(event_list):
    inter_times = [event['time_since_last_event'] for event in event_list]
    inter_times = torch.tensor(inter_times, dtype=torch.float32)
    arrival_times = [event['time_since_start'] for event in event_list]
    type_event = [event['type_event'] for event in event_list]
    type_event = torch.tensor(type_event, dtype=torch.long)
    return EventSequence(
        arrival_times=arrival_times,
        inter_times=inter_times,
        type_event=type_event
    )


In [5]:
data.keys()

dict_keys(['dim_process', 'dev'])

In [6]:
sequence_list = [list_of_dicts_to_sequence(s) for s in data["dev"]]

In [7]:
from src.data.batch import Batch

In [8]:
from src.data.tpp_dataset import TppDataset
ds = TppDataset(sequence_list)
loader = ds.get_dataloader(
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

In [9]:
for batch in loader:
    print(batch.keys())
    break

['arrival_times', 'inter_times', 'non_pad_mask', 'type_seq', 't_start', 't_end']


In [10]:
batch.type_seq[2]

tensor([   8,    3,    8,    3,    8,    3,    8,    3,    8,    3,    8,    3,
           8,    3,    8,    3,    8,    3,    8,    3,    8,    3,    8,    3,
           8,    3,    8,    3,    8,    3,    8,    3,    8,    3,    8,    1,
        -100, -100])

In [11]:
batch.type_seq

tensor([[   8,    0,    5,  ...,    3, -100, -100],
        [   8,    3,    8,  ...,    4,    8,    3],
        [   8,    3,    8,  ...,    1, -100, -100],
        ...,
        [   8,    0,    8,  ...,    3, -100, -100],
        [   5,    3,    8,  ...,    1, -100, -100],
        [   8,    2,    8,  ...,    3,    8,    3]])

In [12]:
batch.arrival_times

tensor([[0.0000, 0.3747, 0.4461,  ..., 4.6194, 0.0000, 0.0000],
        [0.0000, 0.1081, 0.1392,  ..., 9.0400, 9.5700, 9.7367],
        [0.0000, 0.0922, 0.1375,  ..., 6.9478, 0.0000, 0.0000],
        ...,
        [0.0000, 0.2503, 1.2161,  ..., 8.5944, 0.0000, 0.0000],
        [0.0000, 0.5239, 3.0122,  ..., 9.2972, 0.0000, 0.0000],
        [0.0000, 0.4503, 1.0097,  ..., 6.4931, 6.5250, 6.6514]])

In [13]:
batch.inter_times

tensor([[0.0000, 0.3747, 0.0714,  ..., 0.1111, 0.0000, 0.0000],
        [0.0000, 0.1081, 0.0311,  ..., 0.3083, 0.5300, 0.1667],
        [0.0000, 0.0922, 0.0453,  ..., 0.3614, 0.0000, 0.0000],
        ...,
        [0.0000, 0.2503, 0.9658,  ..., 0.3275, 0.0000, 0.0000],
        [0.0000, 0.5239, 2.4883,  ..., 0.3650, 0.0000, 0.0000],
        [0.0000, 0.4503, 0.5594,  ..., 0.1303, 0.0319, 0.1264]])

In [14]:
from config.config_loader import load_args_from_yaml 
args = load_args_from_yaml("config/THP.yaml")
args.dataset = "taxi"
base_dir = f"data/{args.dataset}"

In [15]:
from src.data.preparation import prepare_data_tpp
df, train_loader, val_loader, test_loader,dataset = prepare_data_tpp(
    args,
    base_dir
)

In [16]:
for batch in train_loader:
    print(f"arival_times: {batch.arrival_times}")
    print(f"inter_times: {batch.inter_times}")
    print(f"type_event: {batch.type_seq}")
    print(f"non_pad_mask: {batch.non_pad_mask}")
    print(f"t_start: {batch.t_start}")
    print(f"t_end: {batch.t_end}")  
    break

arival_times: tensor([[ 0.0000,  0.2100,  0.2617,  ...,  9.0100,  0.0000,  0.0000],
        [ 0.0000,  0.0464,  0.0822,  ...,  8.6225, 10.3939, 10.6022],
        [ 0.0000,  0.2036,  0.2489,  ...,  7.1992,  0.0000,  0.0000],
        ...,
        [ 0.0000,  0.1961,  0.2331,  ...,  5.1569,  5.3003,  5.5292],
        [ 0.0000,  0.2081,  0.7561,  ...,  9.7808,  0.0000,  0.0000],
        [ 0.0000,  0.1278,  0.1439,  ...,  7.6128,  7.7614,  7.8633]])
inter_times: tensor([[0.0000, 0.2100, 0.0517,  ..., 0.1050, 0.0000, 0.0000],
        [0.0000, 0.0464, 0.0358,  ..., 0.1319, 1.7714, 0.2083],
        [0.0000, 0.2036, 0.0453,  ..., 0.0353, 0.0000, 0.0000],
        ...,
        [0.0000, 0.1961, 0.0369,  ..., 0.1036, 0.1433, 0.2289],
        [0.0000, 0.2081, 0.5481,  ..., 0.5039, 0.0000, 0.0000],
        [0.0000, 0.1278, 0.0161,  ..., 0.1436, 0.1486, 0.1019]])
type_event: tensor([[   8,    3,    8,  ...,    3, -100, -100],
        [   8,    3,    8,  ...,    3,    8,    1],
        [   8,    3,    8

In [17]:
batch[:,:-3]

EventBatch(
  arrival_times: [32, 35],
  inter_times: [32, 35],
  non_pad_mask: [32, 35],
  type_seq: [32, 35],
  t_start: [32],
  t_end: [32]
)

In [18]:
for batch in train_loader:
   print(
    batch.inter_times.max().item(),
    batch.inter_times.min().item(),
    batch.inter_times.mean().item()
)

   

5.524722099304199 0.0 0.20712420344352722
3.888611078262329 0.0 0.21214592456817627
3.492500066757202 0.0 0.20460230112075806
3.0291666984558105 0.0 0.21236200630664825
3.6997222900390625 0.0 0.20794841647148132
5.111666679382324 0.0 0.21186792850494385
5.349722385406494 0.0 0.22107937932014465


5.061388969421387 0.0 0.21783076226711273
3.0336110591888428 0.0 0.20287716388702393
5.065833568572998 0.0 0.2177739143371582
4.752500057220459 0.0 0.21976448595523834
4.035277843475342 0.0 0.2125493586063385
4.015277862548828 0.0 0.2328798919916153
5.420555591583252 0.0 0.2076219916343689
2.8097221851348877 0.0 0.20471833646297455
3.5627777576446533 0.0 0.20844869315624237
4.513055324554443 0.0 0.216925248503685
3.071666717529297 0.0 0.21468544006347656
3.5063889026641846 0.0 0.21981291472911835
2.2886111736297607 0.0 0.21758794784545898
5.721388816833496 0.0 0.23034697771072388
3.295555591583252 0.0 0.2030971348285675
2.997499942779541 0.0 0.21017704904079437
2.4666666984558105 0.0 0.21965599060058594
4.097222328186035 0.0 0.20630276203155518
3.698333263397217 0.0 0.22502854466438293
3.1455554962158203 0.0 0.20798452198505402
3.5669443607330322 0.0 0.21372190117835999
2.4522221088409424 0.0 0.19661001861095428
3.693333387374878 0.0 0.20331276953220367
3.1844444274902344 0.0 0.2100934

In [19]:
batch.inter_times[0]

tensor([0.0000, 0.2833, 0.1008, 0.4592, 0.0536, 0.1703, 0.0425, 0.3664, 0.7336,
        0.2953, 1.0922, 0.0822, 0.1861, 0.3450, 0.0314, 0.0425, 0.0272, 0.2381,
        0.0167, 0.1972, 0.0781, 0.2392, 0.0528, 0.1703, 0.0283, 0.0914, 0.0272,
        0.1200, 0.0131, 0.1722, 0.0147, 0.3933, 0.3036, 0.1147, 0.3428, 0.1661,
        0.0000, 0.0000])

In [20]:
batch.inter_times[0].max().item()

1.0922222137451172